<a href="https://colab.research.google.com/github/nibaskumar93n-debug/Morphoinformatics/blob/main/ashik_sankey_plot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
uploaded = files.upload()

Saving soil Heavy Metals.xlsx to soil Heavy Metals (3).xlsx


In [3]:
import pandas as pd
import numpy as np

filename = list(uploaded.keys())[0]

if filename.endswith('.csv'):
    df = pd.read_csv(filename)
else:
    df = pd.read_excel(filename)

df.rename(columns={df.columns[0]: 'Sam. No.'}, inplace=True)
print(df.shape)
df.head()

(40, 12)


,Sam. No.,Ni,V,Co,Cd,Fe,Mn,As,Zn,Pb,Cr,Cu
0,ASA-1,32.1,63.8,51.4,0.170000,27946.15,596.548125,7.902725,84.850000,30.285283,121.113281,51.7
1,ASA-2,38.1,68.8,58.4,1.856698,32341.50,638.338235,7.202664,66.887868,37.443623,149.509688,59.1
2,ASA-3,38.9,73.8,61.1,1.000000,33942.60,492.063158,7.325786,77.447368,45.091421,179.843906,58.3
3,ASA-4,36.5,63.8,55.6,0.060000,28941.75,619.801875,7.873208,72.493750,31.396226,125.585156,57.3
4,ASA-5,36.5,68.8,57.0,1.868774,34847.50,644.476103,7.647799,81.590074,37.826818,150.987891,55.5


In [4]:
# Extracts ASA / ND / DUM directly from sample names like ASA-1, ND-21, DUM-31
df['Group'] = df['Sam. No.'].str.extract(r'^([A-Za-z]+)', expand=False).str.upper()
print(df['Group'].value_counts())

Group
ASA    20
ND     10
DUM    10
Name: count, dtype: int64


In [4]:
print(df.columns.tolist())

['Sam. No.', 'Ni', 'V', 'Co', 'Cd', 'Fe', 'Mn', 'As', 'Zn', ' Pb', 'Cr', 'Cu', 'Group']


In [5]:
df.columns = df.columns.str.strip()

In [6]:
metals  = ['Ni', 'V', 'Co', 'Cd', 'Fe', 'Mn', 'As', 'Zn', 'Pb', 'Cr', 'Cu']
sources = ['ASA', 'ND', 'DUM']

means = df.groupby('Group')[metals].mean().loc[sources]
print("Group means (ppm):\n", means.round(2))

pct = means.div(means.sum(axis=0)) * 100
print("\n% contribution per metal:\n", pct.round(1))

Group means (ppm):
           Ni      V     Co    Cd        Fe      Mn    As     Zn     Pb  \
Group                                                                    
ASA    36.82  68.55  54.98  1.06  31444.18  568.75  7.26  72.62  35.48   
ND     48.67  88.30  65.85  1.35  46959.62  740.33  9.27  81.24  38.98   
DUM    45.89  88.30  60.01  1.68  43125.55  675.30  8.49  74.76  35.52   

           Cr     Cu  
Group                 
ASA    141.60  55.05  
ND     155.49  65.76  
DUM    139.31  61.79  

% contribution per metal:
          Ni     V    Co    Cd    Fe    Mn    As    Zn    Pb    Cr    Cu
Group                                                                  
ASA    28.0  28.0  30.4  25.9  25.9  28.7  29.0  31.8  32.3  32.4  30.1
ND     37.0  36.0  36.4  33.1  38.6  37.3  37.0  35.5  35.4  35.6  36.0
DUM    34.9  36.0  33.2  41.1  35.5  34.0  33.9  32.7  32.3  31.9  33.8


In [7]:
import plotly.graph_objects as go

source_colors = {
    'ASA': 'rgba(99,153,34,0.85)',
    'ND':  'rgba(24,95,165,0.85)',
    'DUM': 'rgba(239,159,39,0.85)',
}
metal_colors = {
    'Ni':'rgba(124,58,237,0.85)', 'V' :'rgba(8,145,178,0.85)',
    'Co':'rgba(5,150,105,0.85)',  'Cd':'rgba(217,119,6,0.85)',
    'Fe':'rgba(220,38,38,0.85)',  'Mn':'rgba(219,39,119,0.85)',
    'As':'rgba(79,70,229,0.85)',  'Zn':'rgba(13,148,136,0.85)',
    'Pb':'rgba(234,88,12,0.85)',  'Cr':'rgba(101,163,13,0.85)',
    'Cu':'rgba(147,51,234,0.85)',
}

source_labels = {
    'ASA': 'Agricultural source (ASA)',
    'ND':  'Natural source (ND)',
    'DUM': 'Dumping source (DUM)',
}

all_nodes   = [source_labels[s] for s in sources] + metals
node_colors = [source_colors[s] for s in sources] + \
              [metal_colors[m]  for m in metals]
node_idx    = {n: i for i, n in enumerate(all_nodes)}

link_src, link_tgt, link_val, link_clr = [], [], [], []
for src in sources:
    fade = source_colors[src].replace('0.85', '0.35')
    for metal in metals:
        link_src.append(node_idx[source_labels[src]])
        link_tgt.append(node_idx[metal])
        link_val.append(round(pct.loc[src, metal], 2))
        link_clr.append(fade)

fig = go.Figure(go.Sankey(
    arrangement = 'snap',
    node = dict(
        pad       = 12,
        thickness = 20,
        line      = dict(color='white', width=0.4),
        label     = all_nodes,
        color     = node_colors,
        customdata = all_nodes,
        hovertemplate = '%{customdata}<extra></extra>',
    ),
    link = dict(
        source        = link_src,
        target        = link_tgt,
        value         = link_val,
        color         = link_clr,
        hovertemplate = '%{source.label} → %{target.label}<br>'
                        'Contribution: %{value:.1f}%<extra></extra>',
    ),
))

fig.update_layout(
    title = dict(
        text     = 'Heavy metal source contributions — ASA / ND / DUM samples',
        font     = dict(size=16, family='Arial'),
        x        = 0.5,
        xanchor  = 'center',
    ),
    font          = dict(size=13, family='Arial', color='#333'),
    paper_bgcolor = 'white',
    height        = 580,
    margin        = dict(l=20, r=20, t=60, b=20),
)

fig.show()

/usr/local/lib/python3.12/dist-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




In [8]:
fig.write_image('sankey_heavy_metals.png', width=1400, height=700, scale=2)
print("✓ Saved to session storage")

ValueError: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido


In [9]:
!pip install -U kaleido

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'kaleido'])

import os
os.kill(os.getpid(), 9)   # forces runtime restart

In [10]:
fig.write_image('sankey_heavy_metals.png', width=1400, height=700, scale=2)
print("✓ Saved")

from google.colab import files
files.download('sankey_heavy_metals.png')

ValueError: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
